# Loan Application Evaluator
## Multi-Agent Evaluation System for Financial Sector

This notebook demonstrates a sophisticated loan evaluation system using Strands agents with AWS Bedrock.

### Agents Involved:
1. **Credit Risk Analyst** - Evaluates creditworthiness and credit metrics
2. **Compliance Officer** - Ensures regulatory compliance and documentation
3. **Fraud Detection Specialist** - Identifies fraud patterns and inconsistencies
4. **Loan Officer** - Provides final decision and recommended terms

## Setup and Imports

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime

# Add project to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from loan_evaluator import LoanEvaluator
from models import LoanApplication, EvaluationResult

print("✓ Imports successful")
print(f"Project root: {project_root}")

## Initialize Evaluator

In [ ]:
# Initialize the loan evaluator with AWS Bedrock
evaluator = LoanEvaluator(
    model_name="anthropic.claude-3-5-sonnet-20241022",
    region="us-east-1",
    use_langfuse=False  # Set to True to enable Langfuse tracing
)

print("✓ Evaluator initialized successfully")
print(f"Model: Claude 3.5 Sonnet via Bedrock")
print(f"Agents: {list(evaluator.evaluators.keys())}")

## Load Sample Loan Application

In [ ]:
# Load sample application 1 (Strong applicant)
sample_data_path = Path("sample_data/loan_application_1.json")

with open(sample_data_path, "r") as f:
    app_data = json.load(f)

application = LoanApplication(**app_data)

# Display application summary
print(f"Applicant: {application.applicant_name}")
print(f"Requested Amount: ${application.requested_amount:,.2f}")
print(f"Loan Purpose: {application.loan_purpose}")
print(f"Credit Score: {application.credit_score}")
print(f"Employment: {application.employment_status} at {application.current_employer}")
print(f"Annual Income: ${application.annual_income:,.2f}")

## Run Multi-Agent Evaluation

This will execute all specialized agents and compile their evaluations.

In [ ]:
# Run the evaluation
print("Starting multi-agent evaluation...\n")
result = evaluator.evaluate(application)

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)

## Display Results

In [ ]:
# Summary Results
print(f"\nAPPLICAN: {result.applicant_name}")
print(f"Evaluation Date: {result.evaluation_date}")
print(f"\nFINAL RECOMMENDATION: {result.final_recommendation.upper()}")
print(f"Overall Score: {result.overall_score:.1f}/100")
print(f"Average Confidence: {result.avg_confidence:.1%}")
print(f"\nRationale: {result.decision_rationale}")

## Individual Agent Reviews

In [ ]:
# Detailed agent reviews
for i, review in enumerate(result.reviews, 1):
    print(f"\n{'='*50}")
    print(f"{i}. {review.reviewer_name.upper()}")
    print(f"{'='*50}")
    print(f"Score: {review.score}/100")
    print(f"Recommendation: {review.recommended_action}")
    print(f"Confidence: {review.confidence:.1%}")
    
    print(f"\nStrengths:")
    for strength in review.strengths[:3]:
        print(f"  ✓ {strength}")
    
    print(f"\nWeaknesses:")
    for weakness in review.weaknesses[:3]:
        print(f"  ✗ {weakness}")
    
    print(f"\nRisks:")
    for risk in review.risks[:3]:
        print(f"  ⚠ {risk}")

## Recommended Terms (if approved)

In [ ]:
if result.recommended_interest_rate or result.recommended_loan_amount:
    print("RECOMMENDED LOAN TERMS")
    print("="*50)
    if result.recommended_interest_rate:
        print(f"Interest Rate: {result.recommended_interest_rate:.2%}")
    if result.recommended_loan_amount:
        print(f"Loan Amount: ${result.recommended_loan_amount:,.2f}")
    if result.special_conditions:
        print(f"\nSpecial Conditions:")
        for condition in result.special_conditions:
            print(f"  • {condition}")
else:
    print("No specific terms recommended at this time.")

## Export Results

In [ ]:
# Export the evaluation to JSON
export_path = evaluator.export_result(result)
print(f"Results saved to: {export_path}")

## Test with Additional Applications

In [ ]:
# Load and evaluate additional sample applications
sample_files = list(Path("sample_data").glob("loan_application_*.json"))

print(f"Found {len(sample_files)} sample applications")
print("\nAvailable samples:")
for i, file in enumerate(sorted(sample_files), 1):
    with open(file) as f:
        app_data = json.load(f)
    print(f"{i}. {app_data['applicant_name']} - {app_data['loan_purpose']} (${app_data['requested_amount']:,.0f})")

In [ ]:
# Evaluate another application (e.g., sample 2)
sample_path = Path("sample_data/loan_application_2.json")

with open(sample_path) as f:
    app2_data = json.load(f)

app2 = LoanApplication(**app2_data)

print(f"Evaluating: {app2.applicant_name}")
print(f"Purpose: {app2.loan_purpose}")
print(f"Amount: ${app2.requested_amount:,.2f}\n")

result2 = evaluator.evaluate(app2)
print(f"\nResult: {result2.final_recommendation.upper()} (Score: {result2.overall_score:.1f}/100)")

## Performance Metrics

In [ ]:
# Display evaluation statistics
print("EVALUATION STATISTICS")
print("="*50)
print(f"Number of reviews: {len(result.reviews)}")
print(f"Agents involved: {', '.join(result.agents_involved)}")
print(f"Average confidence: {result.avg_confidence:.1%}")
print(f"Final score distribution:")
print(f"  - Highest score: {max(r.score for r in result.reviews)}/100")
print(f"  - Lowest score: {min(r.score for r in result.reviews)}/100")
print(f"  - Average score: {sum(r.score for r in result.reviews) / len(result.reviews):.1f}/100")